<a href="https://colab.research.google.com/github/jyoti-ai-generalist-portfolio/AIHorizon/blob/main/Batch_Processing_for_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai


Code to do Batch Processing using Open AI API

No, OpenAI's Batch API is not available on the Free Tier. To use the Batch API, your account must be upgraded to at least Tier 1, which requires a minimum prepaid balance deposit of Dollars 5.

The OpenAI Batch API offers a 50 percent discount on input and output tokens and maintains its own separate, higher rate limit pool, but it is strictly restricted to paid usage tiers.


#OpenAI Rate Limit Tiers Overview


The OpenAI Rate Limits Guide structures API access across six automatic spend tiers:


Free Tier: Provided only for initial testing and prototyping with highly restricted rate limits (e.g., 3 requests per minute on core models). Batch API access is completely blocked.


Tier 1: Unlocks instantly upon depositing your first prepaid balance of $5 or more. This grants access to the full model catalog and fully enables the Batch API.


Tiers 2–5: Automatically unlock as your cumulative spend and account age increase, further lifting your synchronous and Batch API rate limits.


How to Activate Batch API Access


If you are currently on the Free Tier and want to run batch jobs, you can upgrade by following these steps:


Navigate to the Billing section within the OpenAI Developer Platform.


Add a valid payment method to your account.


Purchase a minimum of $5 in prepaid API credits.Your account will instantly upgrade to Tier 1, allowing you to upload .jsonl files and execute jobs via the OpenAI Batch API Endpoint.



In [ ]:
import json
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPEN_AI_API_KEY"))

# 1. Create your batch data file locally

tasks = [
    {
        "custom_id": f"task-{i}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o-mini",
            "messages": [{"role": "user", "content": f"Categorize this: {text}"}]
        }
    } for i, text in enumerate(["Text sample A", "Text sample B"])
]

with open("batch_tasks.jsonl", "w") as f:
    for task in tasks:
        f.write(json.dumps(task) + "\n")



# 2. Upload the JSONL file to the AI cloud provider
batch_file = client.files.create(file=open("batch_tasks.jsonl", "rb"), purpose="batch")

# 3. Trigger the asynchronous cloud batch execution
batch_job = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

print(f"Batch job {batch_job.id} is processing.")


BadRequestError: Error code: 400 - {'error': {'message': 'Billing hard limit has been reached', 'type': 'invalid_request_error', 'param': None, 'code': 'billing_hard_limit_reached'}}

Batch processing using Open AI

Batch processing for Gemini API

It is currently restricted for free tier. So code fails with error
<p>
ClientError: 400 FAILED_PRECONDITION. {'error': {'code': 400, 'message': 'Precondition check failed.', 'status': 'FAILED_PRECONDITION'}}

<p>
To be checked in future



In [ ]:
import json
import time
from google import genai
from google.genai import types
from google.colab import userdata

# 1. Initialize the official Google Gen AI Client
# It automatically picks up the GEMINI_API_KEY from environment variables
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# 2. Define the Gemini 3.6 Flash model identifier
MODEL_ID = "gemini-3.6-flash"

# 3. Construct your request batch dataset
texts_to_process = [
    "Classify this support ticket: My account is locked.",
    "Classify this support ticket: I want to request a refund.",
    "Classify this support ticket: App crashes on launch."
]

# Each entry inside the JSONL must mirror a valid generate_content schema
# This should be the format of the JSONL
#{"contents": [{"parts": [{"text": "Summarize transaction history for Account A."}]}]}
#{"contents": [{"parts": [{"text": "Summarize transaction history for Account B."}]}]}



batch_requests = []
for i, text in enumerate(texts_to_process):
    task = {
        "contents": [{"parts": [{"text": text}]}]
    }
    batch_requests.append(task)

# Write out the JSON Lines structure locally
local_filename = "gemini_batch_tasks.jsonl"
with open(local_filename, "w") as f:
    for req in batch_requests:
        f.write(json.dumps(req) + "\n")

print(f"Locally generated payload: {local_filename}")

# 4. Upload your local file directly into Gemini's file service
print("Uploading file to Google AI Studio storage...")
uploaded_file = client.files.upload(file=local_filename,\
config=types.UploadFileConfig(
        display_name="my-batch-requests", mime_type="text/plain"
    ),)

# 5. Initiate the Asynchronous Batch Job
print(f"Creating batch job with {MODEL_ID}...")
print ("Uploaded file name ", uploaded_file.name)
batch_job = client.batches.create( \
    model=MODEL_ID, \
    src=uploaded_file.name , \
    config={
        'display_name': "large-scale-account-summary", \
    }
    # Optional: schema constraints or system instructions can be applied here globally
)

print(f"Batch Job Successfully Created! ID: {batch_job.name}")

# 6. Monitor / Poll Job Processing Status
# (Target completion timeframe is within 24 hours, but often faster)
while True:
    job_status = client.batches.get(name=batch_job.name)
    print(f"Current Status: {job_status.state}")

    if job_status.state == "SUCCEEDED":
        print("Job complete! Ready to download outcomes.")
        # Output URIs will be accessible in job_status.output_file_id
        break
    elif job_status.state in ["FAILED", "CANCELLED"]:
        print(f"Job terminated with unexpected state: {job_status.state}")
        break

    time.sleep(60)  # Check status once per minute

Locally generated payload: gemini_batch_tasks.jsonl
Uploading file to Google AI Studio storage...
Creating batch job with gemini-3.6-flash...
Uploaded file name  files/fw4e4pjrdxsk


ClientError: 400 FAILED_PRECONDITION. {'error': {'code': 400, 'message': 'Precondition check failed.', 'status': 'FAILED_PRECONDITION'}}


#Simulating a "Batch" Pipeline on Gemini's Free Tier

While the formal Batch API is blocked, the Gemini Free Tier is highly lenient, allowing you to build your own synchronous loop or batching script at no cost using their standard endpoints.


You can process data in bulk on the free tier if you manage the pacing yourself:


Generous Daily Quota: Models like Gemini 2.0 Flash provide a high cap of requests per day on the free tier.


Rate Limit Management: You can easily write a Python script that submits requests in parallel or sequentially, using a rate-limiting library (like tenacity or asyncio) to ensure you stay under the free tier limits (typically 10 to 15 Requests Per Minute).


Massive Context Window: Because Gemini Flash supports a 1 million token context window, you don't always need separate API calls. You can combine dozens of smaller tasks into a single prompt, processing multiple data rows in a single free real-time request

Here is the complete Python asyncio script designed to process the McDonald_s_Reviews_Kaggle Dataset.csv using the Google Gen AI Python SDK.

#Architectural Guardrails for the Free Tier

The Gemini Free Tier enforces a maximum daily limit of 250 to 1,000 Requests Per Day (RPD) depending on the exact model version used.

 Because your dataset contains 36,000 records, you cannot process them one-by-one without instantly getting blocked.

 To bypass this barrier completely, this script uses Prompt In-Context Batching:

 It splits your 36,000 rows into structural batch text chunks of 100 reviews each.

 It packs all 100 items into one single API call inside an instruction container.
 This compresses 36,000 records down to just 360 total API calls, safely sliding under daily quotas while executing at peak allowed concurrency.

 Prerequisites

 Install the updated official Google Gen AI SDK along with pandas and tenacity (for automatic retry handling when rate limits are saturated):






In [ ]:
!pip install tenacity asyncio

In [30]:
import asyncio
import os
from google.colab import userdata
import sys
import pandas as pd
from google import genai
from google.genai import types
from google.genai.errors import APIError
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# =====================================================================
# CONFIGURATION & PROMPT SETUP
# =====================================================================
# Customize your baseline instruction prompt here
USER_PROMPT = """
You are an advanced sentiment analysis assistant. Analyze the following list of customer reviews
for McDonald's. For each review, output only its ID and its classified sentiment (Positive, Neutral, or Negative).

Expected output format strictly matching JSON:
[
  {"id": 1, "sentiment": "Positive"},
  {"id": 2, "sentiment": "Negative"}
]
"""

INPUT_FILE = "McDonald_s_Reviews_Kaggle Dataset1.csv"
OUTPUT_FILE = "processed_mcdonalds_reviews.csv"
BATCH_SIZE = 100          # Pack 100 rows into a single prompt payload
CONCURRENT_LIMIT = 2      # Max concurrent requests running at once (Free Tier safe)

# =====================================================================
# INITIALIZE CLIENT & RETRY LOGIC
# =====================================================================
# Use the unified GenAI Client (2026 Standard)
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
# Using gemini-2.5-flash for balanced speed and token efficiency
MODEL_NAME = 'gemini-2.5-flash'
# Semaphore to throttle concurrent asynchronous requests
sem = asyncio.Semaphore(CONCURRENT_LIMIT)

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=2, min=4, max=60),
    retry=retry_if_exception_type(APIError),
    reraise=True
)
async def call_gemini_async(prompt_payload: str) -> str:
    """Wraps the asynchronous content generation call with intelligent backoff retries."""
    async with sem:
        # Utilizing the official async method matching the SDK
        response = await client.aio.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_payload,
            config=types.GenerateContentConfig(
                response_mime_type="application/json" # Enforces clean structure output
            )
        )
        return response.text

# =====================================================================
# BATCH COMPILATION & DISPATCH PIPELINE
# =====================================================================
async def process_batch(batch_df: pd.DataFrame, batch_index: int) -> list:
    """Compiles local data into a text block, fires the prompt, and extracts data rows."""
    print(f"[+] Processing Batch {batch_index}: Processing rows {batch_df.index[0]} to {batch_df.index[-1]}...")

    # Constructing a clean data block inside the prompt
    data_block = "Reviews to analyze:\n"
    for idx, row in batch_df.iterrows():
        # Adjust 'review' column name if your Kaggle CSV uses a different header
        review_text = str(row.get('review', row.iloc[0])).replace('\n', ' ')
        data_block += f"ID: {idx} | Review: {review_text}\n"

    # Combine custom system-level instruction with data block
    full_prompt = f"{USER_PROMPT}\n\n{data_block}"

    try:
        raw_json_response = await call_gemini_async(full_prompt)

        # Convert JSON string directly back into a dataframe segment
        processed_df = pd.read_json(raw_json_response)
        return processed_df.to_dict(orient='records')

    except Exception as e:
        print(f"[-] Critical failure processing batch {batch_index}: {e}", file=sys.stderr)
        # Return partial failure placeholder to avoid breaking the execution loop
        return [{"id": idx, "sentiment": "ERROR_FAILED_ROW"} for idx in batch_df.index]

async def main():
   # if not os.getenv("GEMINI_API_KEY"):
    #    print("[-] Error: GEMINI_API_KEY environment variable is not set.", file=sys.stderr)
     #   return

    print(f"[*] Reading dataset: {INPUT_FILE}...")
    try:
        df = pd.read_csv(INPUT_FILE, encoding_errors='ignore')
    except FileNotFoundError:
        print(f"[-] Error: Could not find file '{INPUT_FILE}' in your running directory.")
        return

    total_records = len(df)
    print(f"[*] Loaded {total_records} rows successfully. Slicing into chunks of {BATCH_SIZE}...")

    # Group records into distinct chunks
    batches = [df[i:i + BATCH_SIZE] for i in range(0, total_records, BATCH_SIZE)]

    # Create the task queue array
    tasks = []
    for idx, batch_df in enumerate(batches):
        tasks.append(process_batch(batch_df, idx + 1))

    print(f"[*] Created {len(tasks)} batch tasks. Executing parallel pipeline...")

    # Run tasks concurrently while respecting the semaphore limit
    results_nested = await asyncio.gather(*tasks)

    # Flatten the array of dictionaries
    flattened_results = [item for sublist in results_nested for item in sublist]

    # Export parsed results back to CSV
    output_df = pd.DataFrame(flattened_results)
    output_df.to_csv(OUTPUT_FILE, index=False)
    print(f"[✓] Complete! All processed outputs saved cleanly to: {OUTPUT_FILE}")

if __name__ == "__main__":
    await main()
   # asyncio.run(main())


[*] Reading dataset: McDonald_s_Reviews_Kaggle Dataset1.csv...
[*] Loaded 33396 rows successfully. Slicing into chunks of 100...
[*] Created 334 batch tasks. Executing parallel pipeline...
[+] Processing Batch 1: Processing rows 0 to 99...
[+] Processing Batch 2: Processing rows 100 to 199...
[+] Processing Batch 3: Processing rows 200 to 299...
[+] Processing Batch 4: Processing rows 300 to 399...
[+] Processing Batch 5: Processing rows 400 to 499...
[+] Processing Batch 6: Processing rows 500 to 599...
[+] Processing Batch 7: Processing rows 600 to 699...
[+] Processing Batch 8: Processing rows 700 to 799...
[+] Processing Batch 9: Processing rows 800 to 899...
[+] Processing Batch 10: Processing rows 900 to 999...
[+] Processing Batch 11: Processing rows 1000 to 1099...
[+] Processing Batch 12: Processing rows 1100 to 1199...
[+] Processing Batch 13: Processing rows 1200 to 1299...
[+] Processing Batch 14: Processing rows 1300 to 1399...
[+] Processing Batch 15: Processing rows 1400

/usr/lib/python3.13/contextlib.py:716: RuntimeWarning: coroutine 'main' was never awaited
  async def __aexit__(self, *exc_details):
[-] Critical failure processing batch 2: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}
[-] Critical failure processing batch 1: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}
[-] Critical failure processing batch 3: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models

[✓] Complete! All processed outputs saved cleanly to: processed_mcdonalds_reviews.csv


[-] Critical failure processing batch 331: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}
[-] Critical failure processing batch 332: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}}
[-] Critical failure processing batch 333: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUN

In [29]:
with open("McDonald_s_Reviews_Kaggle Dataset1.csv", "r", encoding="utf-8") as f:
    content = f.read()
    # 1344th character is at index 1343
    char_1344 = content[1342]
    print(char_1344)

UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 1344-1345: invalid continuation byte